Importación de librería pandas para el trabajo de limpieza y modificación de datos.

In [ ]:
import pandas as pd

Se carga el dataset a utilizar y se presenta una muestra para verificar que fue cargado correctamente.

In [ ]:
# Cargar dataset principal
df = pd.read_csv('Mercado_Chocolate_Dubai.csv')

# Vista inicial de datos
df.head()

,Marca,Producto,Peso_G,Categoria,Version,Precio,%_Pistacho,%_Masa_Kadayif,Pais,Fuente_Producto,Fecha_Deteccion_Mercado,Fuente_Fecha
0,LINDT,TABLETA LINDT DUBAI STYLE CHOCOALTE,150,TABLETA_CHOCOLATE,Estilo_Dubai,"9,99","11,7","2,6",ESPAÑA,https://www.lindt.es/tableta-lindt-dubai-style...,01/12/2024,https://www.revistaaral.com/texto-diario/mostr...
1,LINDT,TABLETA DE CHOCOLATE EXCELLENCE EXTRA CREMOSO ...,100,TABLETA_CHOCOLATE,Clasico,"4,49",0,0,ESPAÑA,https://www.lindt.es/excellence-leche-100g,NaN,NaN
2,HACENDADO,CHOCOLATE CON LECHE FUSSION HACENDADO RELLENO ...,135,TABLETA_CHOCOLATE,Estilo_Dubai,"2,25",15,"7,6",ESPAÑA,https://tienda.mercadona.es/product/16279/choc...,01/02/2026,https://madeinjijona.com/mercadona-lanza-un-nu...
3,HACENDADO,CHOCOLATE CON LECHE FUSSION HACENDADO RELLENO ...,110,TABLETA_CHOCOLATE,Clasico,"1,35",0,0,ESPAÑA,https://tienda.mercadona.es/product/70756/choc...,NaN,NaN
4,TEMPTATION,CHOCOLATE CON PISTACHO ESTILO DUBAI DIA TEMPTA...,150,TABLETA_CHOCOLATE,Estilo_Dubai,"4,49","2,25","2,25",ESPAÑA,https://www.dia.es/azucar-chocolates-y-caramel...,01/11/2025,https://www.consumidorglobal.com/alimentacion/...


Se hace el trabajo de limpieza y modificación de datos, además se agrega una nueva columna. Todo esto con el fin de poder realizar un análisis eficiente de los datos. Al final se presentan tres columnas para verificar la inserción de la nueva columna.

In [ ]:
# 1. Limpieza de la columna Precio (se reemplaza coma por punto).
df['Precio'] = df['Precio'].astype(str).str.replace(',', '.').astype(float)

# 2. Los encabezados de cada columna en minúscula.
df.columns = df.columns.str.lower()

# 3. Cambiamos el formato de fecha para que sea aceptado de manera correcta por SQL.
df['fecha_deteccion_mercado'] = pd.to_datetime(df['fecha_deteccion_mercado'], dayfirst=True, errors='coerce')
df['fecha_deteccion_mercado'] = df['fecha_deteccion_mercado'].dt.to_period('M').dt.to_timestamp()

# 4. Limpieza de porcentajes (los NULL se reemplazan por 0).
df['%_pistacho'] = df['%_pistacho'].astype(str).str.replace(',', '.').replace('NULL', '0').astype(float)
df['%_masa_kadayif'] = df['%_masa_kadayif'].astype(str).str.replace(',', '.').replace('NULL', '0').astype(float)

# 5. Creacion de nueva columna: Precio cada 100 gramos.
df['precio_100g'] = (df['precio'] / df['peso_g']) * 100

df[['marca', 'producto', 'precio_100g']].head()



,marca,producto,precio_100g
0,LINDT,TABLETA LINDT DUBAI STYLE CHOCOALTE,6.660000
1,LINDT,TABLETA DE CHOCOLATE EXCELLENCE EXTRA CREMOSO ...,4.490000
2,HACENDADO,CHOCOLATE CON LECHE FUSSION HACENDADO RELLENO ...,1.666667
3,HACENDADO,CHOCOLATE CON LECHE FUSSION HACENDADO RELLENO ...,1.227273
4,TEMPTATION,CHOCOLATE CON PISTACHO ESTILO DUBAI DIA TEMPTA...,2.993333


El trabajo de limpieza y orden final. Se pule lo realizado y se reordena para una lectura y visualización más eficiente.

In [ ]:
# 1. Corregir fechas (0 por NaN)
df['fecha_deteccion_mercado'] = df['fecha_deteccion_mercado'].replace(0, None)

# 2. Eliminar columnas innecesarias
df = df.drop(columns=['categoria', 'pais', 'fuente_producto', 'fuente_fecha'])

# 3. Reordenar columnas
df = df[['marca', 'producto', 'version', 'peso_g', 'precio', 'precio_100g',
         '%_pistacho', '%_masa_kadayif', 'fecha_deteccion_mercado']]

# 4. Redondear columna de Precio_100g
df['precio_100g'] = df['precio_100g'].round(2)

df.head()

,marca,producto,version,peso_g,precio,precio_100g,%_pistacho,%_masa_kadayif,fecha_deteccion_mercado
0,LINDT,TABLETA LINDT DUBAI STYLE CHOCOALTE,Estilo_Dubai,150,9.99,6.66,11.70,2.60,2024-12-01
1,LINDT,TABLETA DE CHOCOLATE EXCELLENCE EXTRA CREMOSO ...,Clasico,100,4.49,4.49,0.00,0.00,NaT
2,HACENDADO,CHOCOLATE CON LECHE FUSSION HACENDADO RELLENO ...,Estilo_Dubai,135,2.25,1.67,15.00,7.60,2026-02-01
3,HACENDADO,CHOCOLATE CON LECHE FUSSION HACENDADO RELLENO ...,Clasico,110,1.35,1.23,0.00,0.00,NaT
4,TEMPTATION,CHOCOLATE CON PISTACHO ESTILO DUBAI DIA TEMPTA...,Estilo_Dubai,150,4.49,2.99,2.25,2.25,2025-11-01


Se descarga el dataset limpio en formato .csv para luego ser utilizado en el análisis de datos.

In [ ]:
df.to_csv('Mercado_Chocolate_Dubai_Limpio.csv', index=False)